In [1]:
import os
import numpy as np
import glob
from collections import defaultdict
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gsw
from pyproj import Geod
import pandas as pd
import warnings


# Data Import

In [3]:
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="Not a valid ID")

including_models = ["CAS-ESM2-0", 'MPI-ESM1-2-LR']# models not included in calculation
models = []

def dataconcat(scenario, variable):
    dir_path = f"/glade/work/stevenxu/AMOC_models/{variable}/scenarios/{scenario}"
    all_files = glob.glob(os.path.join(dir_path, "*.nc"))

    groups = defaultdict(list)
    for fp in all_files:
        fname = os.path.basename(fp)
        model_name = fname.split("_")[2]
        groups[model_name].append(fp)

    datasets = {}
    for prefix, files in groups.items():
        if prefix not in including_models:
            continue

        files = sorted(files)
        print(f"Testing files for {prefix}")

        clean_files = []
        for f in files:
            # skip zero-byte files immediately
            if os.path.getsize(f) == 0:
                print(f"  Skipping empty file for {prefix}: {f}")
                continue
            try:
                xr.open_dataset(f, engine="netcdf4").close()
                clean_files.append(f)
            except Exception as e:
                print(f"  Skipping unreadable file for {prefix}: {f}")
                print("    Error:", repr(e))

        if not clean_files:
            print(f"⚠ No valid files left for {prefix}, skipping model.")
            continue

        print(f"Concatenating {len(clean_files)} valid files for {prefix}")
        ds = xr.open_mfdataset(
            clean_files,
            combine="by_coords",
            parallel=True,
            engine="netcdf4",
            use_cftime=True,
        )
        datasets[prefix] = ds
        models.append(prefix)

    return datasets
              
sst_datasets = dataconcat("PIControl", "sea_surface_temperature") 
sss_datasets = dataconcat("PIControl", "sea_surface_salinity") 
hf_datasets = dataconcat("PIControl", "heatflux") 
wf_datasets = dataconcat("PIControl", "waterflux")
models = set(models)

Testing files for MPI-ESM1-2-LR
Concatenating 50 valid files for MPI-ESM1-2-LR
Testing files for CAS-ESM2-0
Concatenating 6 valid files for CAS-ESM2-0
Testing files for CAS-ESM2-0
Concatenating 6 valid files for CAS-ESM2-0
Testing files for CAS-ESM2-0
Concatenating 6 valid files for CAS-ESM2-0
Testing files for CAS-ESM2-0
Concatenating 6 valid files for CAS-ESM2-0
Testing files for MPI-ESM1-2-LR
Concatenating 50 valid files for MPI-ESM1-2-LR


In [3]:
def subtract_years_cftime(t, years):
    """Return a new cftime object with years subtracted (same month/day)."""
    return type(t)(
        t.year - years, t.month, t.day,
        t.hour, t.minute, t.second, t.microsecond,
        has_year_zero=t.has_year_zero
    )

def align_time(model, lastNyear):
    # Make sure the model exists everywhere
    for name, dct in [
        ("sst", sst_datasets),
        ("sss", sss_datasets),
        ("hf",  hf_datasets),
        ("wf",  wf_datasets),
    ]:
        if model not in dct:
            print(f"Skipping {model}: missing in {name}_datasets")
            return

    end_times = []
    for variable_ds in [sst_datasets, sss_datasets, hf_datasets, wf_datasets]:
        end_times.append(variable_ds[model]['time'].isel(time=-1).values.item())

    start_time = subtract_years_cftime(min(end_times), lastNyear)
    end_time   = min(end_times)
    print(f"Aligning time for {model} to last {lastNyear} years: {start_time} to {end_time}")

    sst_datasets[model] = sst_datasets[model].sel(time=slice(start_time, end_time))
    sss_datasets[model] = sss_datasets[model].sel(time=slice(start_time, end_time))
    hf_datasets[model]  = hf_datasets[model].sel(time=slice(start_time, end_time))
    wf_datasets[model]  = wf_datasets[model].sel(time=slice(start_time, end_time))


for model in models:
    align_time(model, 20)

Aligning time for CAS-ESM2-0 to last 20 years: 0529-12-16 12:00:00 to 0549-12-16 12:00:00


# Calculate sea surface density

In [4]:
def _get_coord(ds, candidates):
    """
    Return the first coordinate/variable found among candidates from a DataArray or Dataset.
    Works for both coords and data_vars.
    """
    # xarray object could be Dataset or DataArray
    if hasattr(ds, "coords"):
        for name in candidates:
            if name in ds.coords:
                return ds.coords[name]
    if hasattr(ds, "data_vars"):
        for name in candidates:
            if name in ds.data_vars:
                return ds[name]
    # also allow direct indexing for Dataset variables
    for name in candidates:
        if name in ds:
            return ds[name]
    raise KeyError(f"None of these coordinate names exist: {candidates}. "
                   f"Available coords: {list(getattr(ds, 'coords', {}).keys())}. "
                   f"Available vars: {list(getattr(ds, 'variables', {}).keys())[:20]}")

In [5]:
def compute_surface_density(model, sst_datasets, sss_datasets, last_n_months=None):
    t  = sst_datasets[model]["tos"]   # degC (in-situ temp)
    SP = sss_datasets[model]["sos"]   # practical salinity (PSU-like)

    if last_n_months is not None:
        t  = t.isel(time=slice(-last_n_months, None))
        SP = SP.isel(time=slice(-last_n_months, None))

    # lon/lat (1D -> 2D grid)
    lon = sst_datasets[model]["lon"]
    lat = sst_datasets[model]["lat"]
    lon2d, lat2d = xr.broadcast(lon, lat)  # dims (lat, lon)

    p = 0.0  # sea pressure at surface in dbar

    # SP -> SA needs (SP, p, lon, lat)
    SA = xr.apply_ufunc(
        gsw.SA_from_SP, SP, p, lon2d, lat2d,
        dask="parallelized", vectorize=True, output_dtypes=[float],
    )

    # in-situ t -> CT needs (SA, t, p)
    CT = xr.apply_ufunc(
        gsw.CT_from_t, SA, t, p,
        dask="parallelized", vectorize=True, output_dtypes=[float],
    )

    rho = xr.apply_ufunc(
        gsw.rho, SA, CT, p,
        dask="parallelized", vectorize=True, output_dtypes=[float],
    ).rename("rho").assign_attrs(units="kg m-3", long_name="Sea-surface density")

    alpha = xr.apply_ufunc(
        gsw.alpha, SA, CT, p,
        dask="parallelized", vectorize=True, output_dtypes=[float],
    ).rename("alpha")

    beta = xr.apply_ufunc(
        gsw.beta, SA, CT, p,
        dask="parallelized", vectorize=True, output_dtypes=[float],
    ).rename("beta")

    return xr.Dataset(dict(rho=rho, alpha=alpha, beta=beta))

surf_den_CAS = compute_surface_density("CAS-ESM2-0", sst_datasets, sss_datasets, last_n_months=240)
surf_den_CAS

<xarray.Dataset> Size: 406MB
Dimensions:  (time: 240, lat: 196, lon: 360)
Coordinates:
  * time     (time) object 2kB 0530-01-16 12:00:00 ... 0549-12-16 12:00:00
  * lat      (lat) float64 2kB -78.0 -77.0 -76.0 -75.0 ... 87.0 88.0 89.0 90.0
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
Data variables:
    rho      (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>
    alpha    (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>
    beta     (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>

# Calculate F surf

In [6]:
def compute_fsurf(model,
                  sst_datasets, sss_datasets, hf_datasets, wf_datasets,
                  cp=3990.0, rho0=1027.0, rho_fw=1000.0, S0=35.0,
                  last_n_months=None):

    HF = hf_datasets[model]['hfds']  # W m^-2, 
    WF = wf_datasets[model]['wfo']     # kg m^-2 s^-1, 

    density_data = compute_surface_density(model, sst_datasets, sss_datasets, last_n_months=last_n_months)
    rho = density_data['rho']
    alpha = density_data['alpha']
    beta = density_data['beta']

    if last_n_months is not None:
        HF  = HF.isel(time=slice(-last_n_months, None))
        WF = WF.isel(time=slice(-last_n_months, None))

    # f_surf = -(alpha/cp) * f_heat  - (rho0/rho_fw) * beta * S0 * f_water
    fsurf = (alpha / cp) * HF  +  (rho0 / rho_fw) * beta * S0 * WF
    fsurf = fsurf.assign_attrs(
        long_name="Buoyancy-relevant surface forcing (Eq. 5)",
        description="(alpha/cp)*f_heat + (rho0/rho_fw)*beta*S0*f_water",
        units="",
        cp=cp, rho0=rho0, rho_fw=rho_fw, S0=S0
    )

    heat_comp = (alpha / cp) * HF
    fw_comp = (rho0 / rho_fw) * beta * S0 * WF

    return xr.Dataset(dict(fsurf=fsurf, rho=rho, heat_comp=heat_comp, fw_comp=fw_comp))

Fsurf_data = compute_fsurf(
    "CAS-ESM2-0",
    sst_datasets=sst_datasets,
    sss_datasets=sss_datasets,
    hf_datasets=hf_datasets,
    wf_datasets=wf_datasets,
    last_n_months=240
)

Fsurf_datasets = {}
for model in models:
    print(f"Computing F_surf for {model}...")
    Fsurf_datasets[model] = compute_fsurf(
        model,
        sst_datasets=sst_datasets,
        sss_datasets=sss_datasets,
        hf_datasets=hf_datasets,
        wf_datasets=wf_datasets,
         last_n_months=240
    )

Computing F_surf for CAS-ESM2-0...


In [7]:
Fsurf_datasets['CAS-ESM2-0']

<xarray.Dataset> Size: 542MB
Dimensions:    (time: 240, lat: 196, lon: 360)
Coordinates:
  * time       (time) object 2kB 0530-01-16 12:00:00 ... 0549-12-16 12:00:00
  * lat        (lat) float64 2kB -78.0 -77.0 -76.0 -75.0 ... 87.0 88.0 89.0 90.0
  * lon        (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
Data variables:
    fsurf      (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>
    rho        (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>
    heat_comp  (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>
    fw_comp    (time, lat, lon) float64 135MB dask.array<chunksize=(1, 196, 360), meta=np.ndarray>

# Calculating Fgen at single timepoint

### Create density class list
Take max and min in rho values, and slice with interval 0.01

In [8]:
rho_min = 1015
rho_max = 1030

In [ ]:
step_size = 0.005
rho_classes = np.arange(rho_min - step_size, rho_max + step_size, step_size)

### Dataset for area at each grid cell

In [10]:
dir_path = "/glade/work/stevenxu/AMOC_models/areacello"
all_files = glob.glob(os.path.join(dir_path, "*.nc")) 

area_ds = defaultdict(list)
model_names = []
for fp in all_files: 
    fname = os.path.basename(fp) 
    model_name = fname.split("_")[2] 
    area  = xr.open_dataset(fp)["areacello"]
    model_names.append(model_name)
    area_ds[model_name].append(area)
	

### Coordinate Transfer

In [ ]:
def latlon_to_ji(ds, lat_name="lat", lon_name="lon",
                 new_j="j", new_i="i",
                 lat2d_name="latitude", lon2d_name="longitude",
                 drop_1d_latlon=True):
    """
    Convert a rectilinear lat/lon-grid Dataset to j/i-grid style:
      - dims lat/lon -> j/i
      - 1D lat/lon coords -> 2D latitude/longitude coords (j,i)

    This does NOT regrid; it only reshapes/renames to unify conventions.
    """

    # If it's already j/i style, just return
    if (new_j in ds.dims) and (new_i in ds.dims) and (lat2d_name in ds.coords) and (lon2d_name in ds.coords):
        return ds

    # Ensure lat/lon exist (as coords or vars)
    if lat_name in ds.coords:
        lat1d = ds.coords[lat_name]
    else:
        lat1d = ds[lat_name]

    if lon_name in ds.coords:
        lon1d = ds.coords[lon_name]
    else:
        lon1d = ds[lon_name]

    # Rename the dimensions
    rename_map = {}
    if lat_name in ds.dims:
        rename_map[lat_name] = new_j
    if lon_name in ds.dims:
        rename_map[lon_name] = new_i
    ds2 = ds.rename(rename_map)

    # After rename, lat/lon coords might still be named lat/lon but now aligned with j/i dims.
    # Grab the renamed 1D coords aligned to j and i
    lat1d2 = lat1d.rename({lat1d.dims[0]: new_j}) if lat1d.dims else lat1d
    lon1d2 = lon1d.rename({lon1d.dims[0]: new_i}) if lon1d.dims else lon1d

    # Create 2D coords (j,i)
    # xarray broadcast is clean and keeps metadata
    lat2d = lat1d2.broadcast_like(ds2.isel(time=0, drop=True)[next(iter(ds2.data_vars))] if "time" in ds2.dims else next(iter(ds2.data_vars.values())))
    lon2d = lon1d2.broadcast_like(lat2d)

    # Assign 2D coords with standard names
    ds2 = ds2.assign_coords({
        lat2d_name: ( (new_j, new_i), lat2d.data ),
        lon2d_name: ( (new_j, new_i), lon2d.data ),
    })

    # Optionally drop old 1D lat/lon coords if you want everything to look like ACCESS-CM2
    if drop_1d_latlon:
        for n in [lat_name, lon_name]:
            if n in ds2.coords:
                ds2 = ds2.drop_vars(n)

    return ds2


In [ ]:
"""for model, ds in Fsurf_datasets.items():
    # Convert Fsurf dataset lat/lon -> j/i
    if ("lat" in ds.dims) and ("lon" in ds.dims):
        Fsurf_datasets[model] = latlon_to_ji(ds, lat_name="lat", lon_name="lon")

    # Convert areacello lat/lon -> j/i (and keep as list like before)
    area0 = area_ds[model][0]  # DataArray
    if ("lat" in area0.dims) and ("lon" in area0.dims):
        area0_ji = latlon_to_ji(area0.to_dataset(name="areacello"),
                                lat_name="lat", lon_name="lon").areacello
        area_ds[model] = [area0_ji]  """

### Integration

Group by density intervals and adding up the area-weighted fsurf

In [ ]:
Fgen_dict = {} 
for model, Fsurf_data in Fsurf_datasets.items():
    '''
    # build mask on 2D fields
    mask = (_get_coord(Fsurf_data, ["latitude", "lat"]) > 45)

    # stack WITH a MultiIndex (default create_index=True)
    mask_s   = mask.stack(points=("j","i"))

    # keep labels where mask is True (these labels are (j,i) pairs)
    keep_pts = mask_s.where(mask_s, drop=True).coords["points"]

    # stack data and select by labels (sel, not isel)
    fsurf = Fsurf_data["fsurf"].stack(points=("j","i")).sel(points=keep_pts)
    heat_comp = Fsurf_data["heat_comp"].stack(points=("j","i")).sel(points=keep_pts)
    fw_comp = Fsurf_data["fw_comp"].stack(points=("j","i")).sel(points=keep_pts)
    rho   = Fsurf_data["rho"].stack(points=("j","i")).sel(points=keep_pts)
    area1 = area_ds[model][0].stack(points=("j","i")).sel(points=keep_pts)

    # unstack back to (j, i)
    weighted_fsurf = (fsurf * area1).unstack("points").values
    weighted_heat_comp = (heat_comp * area1).unstack("points").values
    weighted_fw_comp = (fw_comp * area1).unstack("points").values
    area_comp = area1.unstack("points").values

    rho = rho.unstack("points").values

    timepoints = Fsurf_data['time'].values'''

    model = 'CAS-ESM2-0'

    Fsurf_data = Fsurf_data.where(
        Fsurf_data.lat > 45,
        drop=True
    )
    fsurf = Fsurf_data["fsurf"]
    heat_comp = Fsurf_data["heat_comp"]
    fw_comp = Fsurf_data["fw_comp"]
    rho   = Fsurf_data["rho"]
    area1 = area_ds[model][0]

    weighted_fsurf      = (fsurf * area1).values
    weighted_heat_comp  = (heat_comp * area1).values
    weighted_fw_comp    = (fw_comp * area1).values
    area_comp = area1.values
    rho = rho.values


    # time: 240  j: 300  i: 360

    Fgen = pd.DataFrame(columns=['time', 'rho', 'Fgen', 'HeatFlux', 'FreshwaterFlux', 'AreaSum'])
    timepoints = Fsurf_data['time'].values
    total_timepoints = len(timepoints)

    for time in range(len(timepoints)):
        for class_idx, rhoclass in enumerate(rho_classes):
            # find range of density
            rhotop = rhoclass + step_size
            rhobot = rhoclass
            # locate density index that are in the range
            rho_idx = np.where((rho[time]>rhobot) & (rho[time]<rhotop))
            inrange_fsurf = []
            inrange_heat_comp = []
            inrange_fw_comp = []
            inrange_area = []
            # looking for all locations that fits the density range
            for lat_idx, lon in enumerate(rho_idx[1]):
                lat = rho_idx[0][lat_idx]
                # adding all valid value into an array
                inrange_fsurf.append(weighted_fsurf[time][lat][lon])
                inrange_heat_comp.append(weighted_heat_comp[time][lat][lon])
                inrange_fw_comp.append(weighted_fw_comp[time][lat][lon])
                inrange_area.append(area_comp[lat][lon])
            # summing up all valid value and process it
            fgen_value = sum(inrange_fsurf) / step_size / 1e6
            heat_comp_value = sum(inrange_heat_comp) / step_size / 1e6
            fw_comp_value = sum(inrange_fw_comp) / step_size / 1e6
            area_sum = sum(inrange_area)
            # outputing
            Fgen.loc[len(Fgen)] = [time, rhoclass + step_size/2, fgen_value, heat_comp_value, fw_comp_value, area_sum]
    Fgen = Fgen.groupby('rho', as_index=False)[['Fgen', 'HeatFlux', 'FreshwaterFlux', 'AreaSum']].mean()
    Fgen_dict[model] = Fgen
    print(f"Completed Fgen calculation for {model}")

KeyboardInterrupt: 

In [ ]:
import pickle

save_path = "/glade/work/stevenxu/AMOC_models/Fgen_Allmodels_norm.pkl"
with open(save_path, "wb") as f:
    pickle.dump(Fgen_dict, f)